# Big Data — Marché Immobilier Français

## 1. Installation et imports

In [1]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'pyspark', '--quiet'])
print('OK')

OK



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os, shutil, re, subprocess, time
import pandas as pd
import re as _re

# Nettoyage des processus Java residuels
subprocess.run('pkill -f SparkSubmit', shell=True, capture_output=True)
subprocess.run('pkill -f pyspark', shell=True, capture_output=True)
subprocess.run('pkill -f GatewayServer', shell=True, capture_output=True)
time.sleep(2)

# Reset interne PySpark
try:
    from pyspark import SparkContext
    if SparkContext._active_spark_context:
        SparkContext._active_spark_context.stop()
    SparkContext._active_spark_context = None
    SparkContext._gateway = None
except Exception:
    pass

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, DoubleType, IntegerType
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName('Immobilier') \
    .config('spark.driver.memory', '2g') \
    .config('spark.sql.shuffle.partitions', '4') \
    .config('spark.ui.enabled', 'false') \
    .config('spark.driver.maxResultSize', '1g') \
    .config('spark.memory.offHeap.enabled', 'true') \
    .config('spark.memory.offHeap.size', '512m') \
    .master('local[2]') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')
DATA_DIR = os.getcwd()
print(f'Spark version : {spark.version}')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/18 14:48:45 WARN Utils: Your hostname, codespaces-d4c346, resolves to a loopback address: 127.0.0.1; using 10.0.1.250 instead (on interface eth0)
26/04/18 14:48:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/18 14:48:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version : 4.1.1


## 2 Nettoyage PAP

Nettoyage du fichier brut `pap_raw.csv` avant ingestion Spark :
- Séparation `type` → `transaction` + `type_bien`
- Extraction `ville` et `code_postal` depuis la colonne brute
- Conversion des types numériques
- Export `pap_clean.csv`

In [4]:
# Chargement du fichier
pap_raw_path = os.path.join(DATA_DIR, 'pap_raw.csv')
df_pap_brut = pd.read_csv(pap_raw_path, on_bad_lines='skip', sep=None, engine='python')

# Nettoyage des noms de colonnes 
df_pap_brut.columns = [c.strip().strip('\ufeff').strip('"') for c in df_pap_brut.columns]
print('Colonnes detectees :', list(df_pap_brut.columns))
print(f'Lignes brutes : {len(df_pap_brut)}')

Colonnes detectees : ['type', 'ville', 'prix', 'surface', 'nb_pieces', 'nb_chambres', 'description']
Lignes brutes : 1470


In [5]:
# Séparation du type en deux colonnes (transaction, type_bien)
df_pap_brut[['transaction', 'type_bien']] = df_pap_brut['type'].str.split('-', n=1, expand=True)
df_pap_brut.drop(columns=['type'], inplace=True)

# Ajout de la source
df_pap_brut['source'] = 'pap'

# Extraction de la ville et du code postal 
def extract_ville_cp(s):
    if pd.isna(s):
        return pd.NA, pd.NA
    s = s.replace('\xa0', ' ')
    cp_match = _re.search(r'\((\d{5})\)', s)
    cp = cp_match.group(1) if cp_match else pd.NA
    ville_match = _re.search(r'([\w-]+)\s+([\w-]+)\s*\(\d{5}\)', s)
    if ville_match:
        avant_dernier = ville_match.group(1)
        dernier = ville_match.group(2)
        ville = avant_dernier if _re.match(r'^\d+\w*$', dernier) else dernier
    else:
        m = _re.search(r'([\w-]+)\s*\(\d{5}\)', s)
        ville = m.group(1) if m else pd.NA
    return ville, cp

df_pap_brut[['ville_clean', 'code_postal']] = df_pap_brut['ville'].apply(
    lambda s: pd.Series(extract_ville_cp(s))
)
df_pap_brut.drop(columns=['ville'], inplace=True)
df_pap_brut.rename(columns={'ville_clean': 'ville'}, inplace=True)

# Conversion des types 
df_pap_brut['surface'] = pd.to_numeric(df_pap_brut['surface'], errors='coerce')
for col in ['nb_pieces', 'nb_chambres', 'prix']:
    df_pap_brut[col] = pd.to_numeric(df_pap_brut[col], errors='coerce').astype('Int64')

# Réordonner les colonnes
cols_ordre = ['transaction', 'type_bien', 'prix', 'surface',
              'nb_pieces', 'nb_chambres', 'ville', 'code_postal', 'description', 'source']
cols_ordre = [c for c in cols_ordre if c in df_pap_brut.columns]
df_pap_brut = df_pap_brut[cols_ordre]

print(f'Lignes apres nettoyage : {len(df_pap_brut)}')
print(df_pap_brut.dtypes)
print()
print(df_pap_brut.head(5).to_string())

Lignes apres nettoyage : 1470
transaction     object
type_bien       object
prix             Int64
surface        float64
nb_pieces        Int64
nb_chambres      Int64
ville           object
code_postal     object
description     object
source          object
dtype: object

  transaction     type_bien    prix  surface  nb_pieces  nb_chambres  ville code_postal                                                                                                                                                                                                description source
0      ventes  appartements  610000    65.00          3            2  Paris       75010   3 pièces de 65 m², au 154 rue La Fayette  au croisement entre le rue La Fayette et la rue du Faubourg St-Denis,  2 min du métro Gare du Nord.  - 5eme étage avec ascenseur, appartement très lumineux...    pap
1      ventes  appartements  532000    65.00          3            2  Paris       75013  Paris 13 ème – Quartier Jeanne d'Arc.  D

In [6]:
# Export du dataframe
pap_clean_path = os.path.join(DATA_DIR, 'pap_clean.csv')
df_pap_brut.to_csv(pap_clean_path, sep=';', index=False, encoding='utf-8-sig', quoting=1)
print(f'\npap_clean.csv exporté : {len(df_pap_brut)} lignes')


pap_clean.csv exporté : 1470 lignes


## 3. Chargement des données

`pap_clean.csv` est généré par l'étape 2, `orpi_raw.json` et `entreparticuliers_raw.json` sont issus du scraping.

In [7]:
# PAP
df_pap_raw = spark.read \
    .option('header', 'true') \
    .option('sep', ';') \
    .option('encoding', 'UTF-8') \
    .csv(os.path.join(DATA_DIR, 'pap_clean.csv'))
print(f'PAP : {df_pap_raw.count()} lignes')

# Orpi
df_orpi_raw = spark.read \
    .option('multiline', 'true') \
    .json(os.path.join(DATA_DIR, 'orpi_raw.json'))
print(f'Orpi : {df_orpi_raw.count()} lignes')

# Entreparticuliers
df_ep_raw = spark.read \
    .option('multiline', 'true') \
    .json(os.path.join(DATA_DIR, 'entreparticuliers_raw (1).json'))
print(f'EP : {df_ep_raw.count()} lignes')

PAP : 1470 lignes
Orpi : 544 lignes
EP : 480 lignes


## 4. Harmonisation et fusion des tables

In [8]:
# PAP
df_pap = df_pap_raw \
    .withColumnRenamed('transaction', 'type_transaction') \
    .withColumn('type_transaction',
        F.when(F.col('type_transaction').isin('ventes','vente'), 'vente')
         .when(F.col('type_transaction').isin('locations','location'), 'location')
         .otherwise(F.col('type_transaction'))) \
    .withColumn('type_bien',
        F.when(F.lower('type_bien').isin('appartements','appartement'), 'Appartement')
         .when(F.lower('type_bien').isin('maisons','maison'), 'Maison')
         .when(F.lower('type_bien').isin('studios','studio'), 'Studio')
         .when(F.lower('type_bien').isin('villas','villa'), 'Villa')
         .otherwise(F.initcap('type_bien'))) \
    .withColumn('prix',        F.col('prix').cast(DoubleType())) \
    .withColumn('surface',     F.col('surface').cast(DoubleType())) \
    .withColumn('nb_pieces',   F.col('nb_pieces').cast(DoubleType())) \
    .withColumn('nb_chambres', F.col('nb_chambres').cast(DoubleType())) \
    .withColumn('source',      F.lit('pap')) \
    .select('type_transaction','type_bien','prix','surface',
            'nb_pieces','nb_chambres','ville','code_postal','description','source')
print(f'PAP harmonisé : {df_pap.count()} lignes')

# Orpi
df_orpi = df_orpi_raw \
    .withColumn('prix',        F.col('prix').cast(DoubleType())) \
    .withColumn('surface',     F.col('surface').cast(DoubleType())) \
    .withColumn('nb_pieces',   F.col('nb_pieces').cast(DoubleType())) \
    .withColumn('nb_chambres', F.lit(None).cast(DoubleType())) \
    .select('type_transaction','type_bien','prix','surface',
            'nb_pieces','nb_chambres','ville','code_postal','description','source')
print(f'Orpi harmonisé : {df_orpi.count()} lignes')

# Entreparticuliers
df_ep = df_ep_raw \
    .withColumn('prix',        F.col('prix').cast(DoubleType())) \
    .withColumn('surface',     F.col('surface').cast(DoubleType())) \
    .withColumn('nb_pieces',   F.col('nb_pieces').cast(DoubleType())) \
    .withColumn('nb_chambres', F.lit(None).cast(DoubleType())) \
    .withColumn('description',
        F.when(F.col('description') == '', None).otherwise(F.col('description'))) \
    .select('type_transaction','type_bien','prix','surface',
            'nb_pieces','nb_chambres','ville','code_postal','description','source')
print(f'EP harmonisé : {df_ep.count()} lignes')

# Fusion
df = df_pap.union(df_orpi).union(df_ep)
print(f'Total fusionné : {df.count()} lignes')
df.printSchema()

PAP harmonisé : 1470 lignes
Orpi harmonisé : 544 lignes
EP harmonisé : 480 lignes
Total fusionné : 2494 lignes
root
 |-- type_transaction: string (nullable = true)
 |-- type_bien: string (nullable = true)
 |-- prix: double (nullable = true)
 |-- surface: double (nullable = true)
 |-- nb_pieces: double (nullable = true)
 |-- nb_chambres: double (nullable = true)
 |-- ville: string (nullable = true)
 |-- code_postal: string (nullable = true)
 |-- description: string (nullable = true)
 |-- source: string (nullable = true)



In [9]:
# Aperçu du dataset fusionné
print(f'Dataset fusionné : {df.count()} lignes')
df.groupBy('source','type_transaction').count().orderBy('source').show()
df.show(5, truncate=60)

Dataset fusionné : 2494 lignes
+-----------------+----------------+-----+
|           source|type_transaction|count|
+-----------------+----------------+-----+
|entreparticuliers|        location|  378|
|entreparticuliers|           vente|  102|
|             orpi|           vente|  269|
|             orpi|        location|  275|
|              pap|           vente|  726|
|              pap|        location|  744|
+-----------------+----------------+-----+



+----------------+-----------+--------+-------+---------+-----------+-----+-----------+------------------------------------------------------------+------+
|type_transaction|  type_bien|    prix|surface|nb_pieces|nb_chambres|ville|code_postal|                                                 description|source|
+----------------+-----------+--------+-------+---------+-----------+-----+-----------+------------------------------------------------------------+------+
|           vente|Appartement|610000.0|   65.0|      3.0|        2.0|Paris|      75010|3 pièces de 65 m², au 154 rue La Fayette  au croisement e...|   pap|
|           vente|Appartement|532000.0|   65.0|      3.0|        2.0|Paris|      75013|Paris 13 ème – Quartier Jeanne d'Arc.  Dans une résidence...|   pap|
|           vente|Appartement|128000.0|   9.31|      1.0|        1.0|Paris|      75005|Chambre Métro Maubert-Mutualité.  Au 6ᵉ étage d'un immeub...|   pap|
|           vente|Appartement|330000.0|   28.0|      2.0|       

## 5. Nettoyage

In [11]:
# Suppression nulls + filtre transactions
from pyspark.ml.feature import Imputer
avant = df.count()
df = df.dropna(subset=['prix', 'surface'])
df = df.filter(F.col('type_transaction').isin('vente', 'location'))

# Imputer necessite DoubleType
df = df.withColumn('nb_pieces', F.col('nb_pieces').cast(DoubleType()))
imputer = Imputer(inputCols=['nb_pieces'], outputCols=['nb_pieces'], strategy='median')
df = imputer.fit(df).transform(df)

df = df.withColumn('ville', F.initcap(F.trim(F.col('ville')))) \
       .withColumn('code_postal', F.trim(F.col('code_postal')))
print(f'Apres nulls  : {df.count()} lignes (supprime {avant - df.count()})') 

Apres nulls  : 2416 lignes (supprime 0)


In [12]:
# Deduplication
avant = df.count()
window_dedup = Window.partitionBy(
    'type_transaction','type_bien','prix','surface','ville'
).orderBy(F.col('description').isNull().asc())

df = df.withColumn('_rank', F.row_number().over(window_dedup)) \
       .filter(F.col('_rank') == 1) \
       .drop('_rank')
print(f'Apres dedup : {df.count()} lignes (supprime {avant - df.count()})') 

Apres dedup : 1844 lignes (supprime 572)


In [13]:
# Filtrage outliers IQR x3
def filter_iqr(df, col_name, group_col='type_transaction', factor=3.0):
    quantiles = df.groupBy(group_col).agg(
        F.expr(f'percentile_approx({col_name}, 0.25)').alias('q1'),
        F.expr(f'percentile_approx({col_name}, 0.75)').alias('q3')
    ).withColumn('iqr',   F.col('q3') - F.col('q1')) \
     .withColumn('lower', F.col('q1') - factor * F.col('iqr')) \
     .withColumn('upper', F.col('q3') + factor * F.col('iqr')) \
     .select(group_col, 'lower', 'upper')
    quantiles.show()
    return df.join(quantiles, on=group_col, how='left') \
             .filter((F.col(col_name) >= F.col('lower')) &
                     (F.col(col_name) <= F.col('upper'))) \
             .drop('lower', 'upper')

avant = df.count()
df.cache()
df.count()

df = filter_iqr(df, 'prix')
df = filter_iqr(df, 'surface')
df = df.filter((F.col('surface') >= 5) & (F.col('surface') <= 1000))
df = df.filter(F.col('prix') > 0)
df.cache()
df.count()
print(f'Apres IQR : {df.count()} lignes (supprime {avant - df.count()})') 

+----------------+---------+---------+
|type_transaction|    lower|    upper|
+----------------+---------+---------+
|        location|  -1670.0|   3650.0|
|           vente|-674000.0|1433000.0|
+----------------+---------+---------+

+----------------+------+-----+
|type_transaction| lower|upper|
+----------------+------+-----+
|        location|-121.0|229.0|
|           vente|-148.0|321.0|
+----------------+------+-----+

Apres IQR : 1790 lignes (supprime 54)


In [14]:
# Bilan
print('=' * 50)
print('BILAN — DATASET NETTOYE')
print('=' * 50)
df.groupBy('source', 'type_transaction').count().orderBy('source').show()
df.select('prix', 'surface', 'nb_pieces').describe().show()
n_desc = df.filter(F.col('description').isNotNull()).count()
print(f'Avec description : {n_desc} / {df.count()} ({n_desc/df.count()*100:.1f}%)')

BILAN — DATASET NETTOYE
+-----------------+----------------+-----+
|           source|type_transaction|count|
+-----------------+----------------+-----+
|entreparticuliers|        location|  197|
|entreparticuliers|           vente|   87|
|             orpi|        location|  274|
|             orpi|           vente|  264|
|              pap|        location|  365|
|              pap|           vente|  603|
+-----------------+----------------+-----+

+-------+------------------+-----------------+------------------+
|summary|              prix|          surface|         nb_pieces|
+-------+------------------+-----------------+------------------+
|  count|              1790|             1790|              1790|
|   mean|215975.28547486034|76.84949720670392| 3.170949720670391|
| stddev| 278388.9333267077|50.05262172344656|1.6211447599932192|
|    min|              70.0|             7.96|               1.0|
|    max|         1390000.0|            304.0|              12.0|
+-------+--------

## 6. NLP MapReduce

In [15]:
STOP_WORDS = {
    'le','la','les','de','du','des','un','une','et','en','au','aux','a','l',
    'se','sa','son','sur','sous','par','pour','avec','dans','est','ce','qui',
    'que','il','elle','ils','elles','vous','nous','on','je','tu','y','ou',
    'si','mais','donc','car','ni','ne','pas','plus','d','qu','n','c','j',
    'tres','bien','tout','cette','cet','ses','leur','leurs'
}
stop_bc = spark.sparkContext.broadcast(STOP_WORDS)

def tokenize(text):
    stop = stop_bc.value
    words = re.sub(r'[^a-z\s]', ' ', text.lower()).split()
    return [(w, 1) for w in words if len(w) > 2 and w not in stop]

descriptions_rdd = df.filter(F.col('description').isNotNull()) \
                     .select('description') \
                     .rdd.map(lambda r: r['description'])

K = 30
top_k = descriptions_rdd.flatMap(tokenize) \
                         .reduceByKey(lambda a, b: a + b) \
                         .sortBy(lambda x: x[1], ascending=False) \
                         .take(K)

print(f'Top {K} mots (RDD MapReduce) :')
print(f'{"Rang":<6} {"Mot":<20} {"Freq":>6}')
print('-' * 35)
for i, (mot, freq) in enumerate(top_k, 1):
    print(f'{i:<6} {mot:<20} {freq:>6}')

Top 30 mots (RDD MapReduce) :
Rang   Mot                    Freq
-----------------------------------
1      appartement             695
2      maison                  666
3      situ                    546
4      tage                    376
5      calme                   329
6      cuisine                 307
7      salle                   290
8      meubl                   259
9      ces                     257
10     quartier                220
11     chambres                217
12     centre                  205
13     proximit                197
14     ville                   191
15     quip                    179
16     nov                     179
17     commerces               178
18     jour                    177
19     lumineux                176
20     terrain                 175
21     pied                    170
22     chambre                 170
23     rement                  169
24     rue                     166
25     louer                   164
26     enti             

In [16]:
# Top 15 mots par type de transaction
for transaction in ['vente', 'location']:
    print(f'\n--- Top 15 mots — {transaction.upper()} ---')
    df.filter(
        F.col('description').isNotNull() &
        (F.col('type_transaction') == transaction)
    ).select(
        F.explode(
            F.split(F.regexp_replace(F.lower('description'), r'[^a-z\s]', ' '), r'\s+')
        ).alias('mot')
    ).filter(F.length('mot') > 2) \
     .filter(~F.col('mot').isin(list(STOP_WORDS))) \
     .groupBy('mot').agg(F.count('*').alias('frequence')) \
     .orderBy(F.desc('frequence')) \
     .limit(15).show(truncate=False)


--- Top 15 mots — VENTE ---
+-----------+---------+
|mot        |frequence|
+-----------+---------+
|maison     |425      |
|appartement|352      |
|situ       |333      |
|tage       |206      |
|calme      |198      |
|terrain    |153      |
|ces        |150      |
|quartier   |129      |
|vendre     |116      |
|proximit   |110      |
|chambres   |104      |
|commerces  |99       |
|lumineux   |99       |
|couvrez    |98       |
|salle      |97       |
+-----------+---------+


--- Top 15 mots — LOCATION ---
+-----------+---------+
|mot        |frequence|
+-----------+---------+
|appartement|343      |
|maison     |241      |
|meubl      |238      |
|situ       |213      |
|cuisine    |211      |
|salle      |193      |
|tage       |170      |
|louer      |164      |
|quip       |138      |
|calme      |131      |
|ville      |120      |
|chambre    |116      |
|chambres   |113      |
|centre     |112      |
|eau        |111      |
+-----------+---------+



## 7. Prix au m²

In [17]:
df = df.withColumn('prix_m2', F.col('prix') / F.col('surface'))
df = df.filter(
    F.col('prix_m2').isNotNull() &
    (F.col('prix_m2') > 200)    &
    (F.col('prix_m2') < 30000)  &
    (F.col('surface') >= 9)
)
print(f'Lignes apres filtrage prix_m2 : {df.count()}')

df.groupBy('type_bien') \
  .agg(F.count('*').alias('nb'),
       F.round(F.mean('prix_m2'), 0).alias('prix_m2_moyen'),
       F.round(F.min('prix_m2'),  0).alias('min'),
       F.round(F.max('prix_m2'),  0).alias('max')) \
  .orderBy(F.desc('prix_m2_moyen')).show()

df.groupBy('type_transaction') \
  .agg(F.count('*').alias('nb'),
       F.round(F.mean('prix_m2'), 0).alias('prix_m2_moyen')).show()

Lignes apres filtrage prix_m2 : 938
+-----------+---+-------------+------+-------+
|  type_bien| nb|prix_m2_moyen|   min|    max|
+-----------+---+-------------+------+-------+
|Appartement|468|       6558.0|1013.0|24074.0|
|     Maison|468|       3780.0| 506.0|13906.0|
|           |  2|        764.0| 750.0|  779.0|
+-----------+---+-------------+------+-------+

+----------------+---+-------------+
|type_transaction| nb|prix_m2_moyen|
+----------------+---+-------------+
|           vente|938|       5160.0|
+----------------+---+-------------+



## 8. Feature Engineering

In [19]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
df = df.withColumn('nb_pieces',   F.coalesce(F.col('nb_pieces'),   F.lit(3.0))) \
       .withColumn('nb_chambres', F.coalesce(F.col('nb_chambres'), F.lit(1.0)))

df = df.withColumn(
    'departement',
    F.when(F.col('code_postal').isNotNull(),
           F.substring(F.col('code_postal').cast('string'), 1, 2)
    ).otherwise('00')
)

# Remplacement chaines vides — StringIndexer refuse les ""
cat_cols = ['type_bien', 'type_transaction', 'source', 'departement']
for c in cat_cols:
    df = df.withColumn(c,
        F.when((F.col(c).isNull()) | (F.trim(F.col(c)) == ''), F.lit('inconnu'))
         .otherwise(F.col(c)))

num_cols  = ['surface', 'nb_pieces', 'nb_chambres']
indexers  = [StringIndexer(inputCol=c, outputCol=c+'_idx', handleInvalid='keep') for c in cat_cols]
encoders  = [OneHotEncoder(inputCol=c+'_idx', outputCol=c+'_ohe') for c in cat_cols]
assembler = VectorAssembler(
    inputCols=num_cols + [c+'_ohe' for c in cat_cols],
    outputCol='features', handleInvalid='skip'
)
print('Features :', num_cols + cat_cols)
print('Target : prix_m2')

Features : ['surface', 'nb_pieces', 'nb_chambres', 'type_bien', 'type_transaction', 'source', 'departement']
Target : prix_m2


## 9. Split Train / Test

In [20]:
df_ml = df.select(
    *num_cols, *cat_cols,
    'prix_m2', 'prix', 'ville', 'code_postal'
).dropna(subset=['prix_m2'])

train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)

# Cache obligatoire — evite de recalculer tout le pipeline a chaque modele
train_df.cache()
test_df.cache()
train_df.count()
test_df.count()

print(f'Train : {train_df.count()} lignes (cache)')
print(f'Test  : {test_df.count()} lignes (cache)')

Train : 755 lignes (cache)
Test  : 183 lignes (cache)


## 10. Entraînement des modèles

In [26]:
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.ml import Pipeline

evaluator      = RegressionEvaluator(labelCol='prix_m2', predictionCol='prediction', metricName='r2')
evaluator_rmse = RegressionEvaluator(labelCol='prix_m2', predictionCol='prediction', metricName='rmse')

results = []

# Regression Lineaire 
lr = LinearRegression(labelCol='prix_m2', featuresCol='features', maxIter=50)
model_lr = Pipeline(stages=indexers + encoders + [assembler, lr]).fit(train_df)
preds_lr = model_lr.transform(test_df)
r2_lr   = evaluator.evaluate(preds_lr)
rmse_lr = evaluator_rmse.evaluate(preds_lr)
preds_lr.unpersist()
print(f'Regression Lineaire — R2={r2_lr:.4f} | RMSE={rmse_lr:.2f} euros/m2')
results.append(('Regression Lineaire', r2_lr, rmse_lr, model_lr))

# Random Forest 
rf = RandomForestRegressor(labelCol='prix_m2', featuresCol='features',
                           numTrees=30, maxDepth=4, seed=42)
model_rf = Pipeline(stages=indexers + encoders + [assembler, rf]).fit(train_df)
preds_rf = model_rf.transform(test_df)
r2_rf   = evaluator.evaluate(preds_rf)
rmse_rf = evaluator_rmse.evaluate(preds_rf)
preds_rf.unpersist()
print(f'Random Forest       — R2={r2_rf:.4f} | RMSE={rmse_rf:.2f} euros/m2')
results.append(('Random Forest', r2_rf, rmse_rf, model_rf))

# GBT 
gbt = GBTRegressor(labelCol='prix_m2', featuresCol='features',
                   maxIter=15, maxDepth=3, seed=42)
model_gbt = Pipeline(stages=indexers + encoders + [assembler, gbt]).fit(train_df)
preds_gbt = model_gbt.transform(test_df)
r2_gbt   = evaluator.evaluate(preds_gbt)
rmse_gbt = evaluator_rmse.evaluate(preds_gbt)
preds_gbt.unpersist()
print(f'GBT                 — R2={r2_gbt:.4f} | RMSE={rmse_gbt:.2f} euros/m2')
results.append(('GBT', r2_gbt, rmse_gbt, model_gbt))

Regression Lineaire — R2=0.6728 | RMSE=1928.85 euros/m2
Random Forest       — R2=0.6486 | RMSE=1999.04 euros/m2
GBT                 — R2=0.6727 | RMSE=1929.29 euros/m2


## 11. Comparaison des modèles

In [27]:
print(f'{"Modele":<25} {"R2":>8} {"RMSE":>12}')
print('-' * 48)
best_r2 = max(r[1] for r in results)
for name, r2, rmse, _ in sorted(results, key=lambda x: -x[1]):
    print(f'{name:<25} {r2:>8.4f} {rmse:>12.2f}')

best_name, best_r2, best_rmse, best_model = max(results, key=lambda x: x[1])
print(f'\nModele selectionne : {best_name}')

Modele                          R2         RMSE
------------------------------------------------
Regression Lineaire         0.6728      1928.85
GBT                         0.6727      1929.29
Random Forest               0.6486      1999.04

Modele selectionne : Regression Lineaire


## 12. Analyse des tendances

In [28]:
print('Top 20 villes — Prix/m2 moyen (vente)')
df.filter(F.col('type_transaction') == 'vente') \
  .groupBy('ville') \
  .agg(F.count('*').alias('nb'), F.round(F.mean('prix_m2'), 0).alias('prix_m2_moyen')) \
  .filter(F.col('nb') >= 5) \
  .orderBy(F.desc('prix_m2_moyen')) \
  .show(20, truncate=False)

print('\nPrix/m2 par type de bien et transaction')
df.groupBy('type_bien', 'type_transaction') \
  .agg(F.count('*').alias('nb'), F.round(F.mean('prix_m2'), 0).alias('prix_m2_moyen')) \
  .orderBy('type_bien', 'type_transaction').show()

Top 20 villes — Prix/m2 moyen (vente)
+-------------------+---+-------------+
|ville              |nb |prix_m2_moyen|
+-------------------+---+-------------+
|Paris              |166|10821.0      |
|Saint-cloud        |6  |8165.0       |
|Garches            |8  |6542.0       |
|Rueil-malmaison    |8  |6528.0       |
|Maisons-alfort     |8  |6484.0       |
|Châtenay-malabry   |5  |5433.0       |
|Marseille          |12 |5387.0       |
|Antony             |5  |4879.0       |
|Lyon               |5  |4846.0       |
|Haÿ-les-roses      |11 |4808.0       |
|Villejuif          |6  |4762.0       |
|Neuilly-plaisance  |11 |4497.0       |
|Champigny-sur-marne|12 |4193.0       |
|Épinay-sur-seine   |5  |4051.0       |
|Vitry-sur-seine    |10 |4029.0       |
|Lille              |9  |4015.0       |
|Strasbourg         |5  |4008.0       |
|Argenteuil         |9  |3867.0       |
|Créteil            |6  |3790.0       |
|Toulouse           |9  |3629.0       |
+-------------------+---+-------------+
on

In [ ]:
# Rendement locatif brut
vente_df = df.filter(F.col('type_transaction') == 'vente') \
    .groupBy('ville') \
    .agg(F.count('*').alias('nb_ventes'), F.mean('prix').alias('prix_vente_moyen')) \
    .filter(F.col('nb_ventes') >= 3)

location_df = df.filter(F.col('type_transaction') == 'location') \
    .groupBy('ville') \
    .agg(F.count('*').alias('nb_locations'), F.mean('prix').alias('loyer_moyen')) \
    .filter(F.col('nb_locations') >= 3)

vente_df.join(location_df, on='ville', how='inner') \
    .withColumn('rendement_brut_pct',
        F.round((F.col('loyer_moyen') * 12) / F.col('prix_vente_moyen') * 100, 2)) \
    .filter(F.col('rendement_brut_pct') > 0) \
    .orderBy(F.desc('rendement_brut_pct')) \
    .select('ville','nb_ventes','nb_locations',
            F.round('prix_vente_moyen',0).alias('prix_vente_moyen'),
            F.round('loyer_moyen',0).alias('loyer_moyen'),
            'rendement_brut_pct') \
    .show(15, truncate=False)

+-----+---------+------------+----------------+-----------+------------------+
|ville|nb_ventes|nb_locations|prix_vente_moyen|loyer_moyen|rendement_brut_pct|
+-----+---------+------------+----------------+-----------+------------------+
+-----+---------+------------+----------------+-----------+------------------+



## 13. Scoring des biens

In [ ]:
df_scored = best_model.transform(df_ml) \
    .withColumn('prix_m2_predit', F.round(F.col('prediction'), 2)) \
    .withColumn('ecart_pct',
        F.round((F.col('prix_m2') - F.col('prediction')) / F.col('prediction') * 100, 2)) \
    .withColumn('score_marche',
        F.when(F.col('ecart_pct') < -15, 'Sous-evalue')
         .when(F.col('ecart_pct') >  15, 'Sur-evalue')
         .otherwise('Prix marche'))

print('Distribution')
df_scored.groupBy('score_marche').count().orderBy('count').show()

df_scored.select(
    'ville','type_bien',
    F.round('surface',1).alias('surface'),
    F.round('prix_m2',2).alias('prix_m2_reel'),
    'prix_m2_predit','ecart_pct','score_marche'
).show(10, truncate=40)

=== Distribution ===
+------------+-----+
|score_marche|count|
+------------+-----+
|  Sur-evalue|  205|
| Sous-evalue|  272|
| Prix marche|  344|
+------------+-----+

+-----------------+---------+-------+------------+--------------+---------+------------+
|            ville|type_bien|surface|prix_m2_reel|prix_m2_predit|ecart_pct|score_marche|
+-----------------+---------+-------+------------+--------------+---------+------------+
|      Saint-cloud|   Maison|  170.0|     8588.24|       6314.31|    36.01|  Sur-evalue|
|          Grimaud|   Maison|  215.0|     6465.12|       3394.41|    90.46|  Sur-evalue|
|           Hyères|   Maison|  165.0|     8424.24|       2564.03|   228.55|  Sur-evalue|
|          Garches|   Maison|  150.0|     7266.67|       6402.06|    13.51| Prix marche|
|       Courbevoie|   Maison|  120.0|     7908.33|       6402.06|    23.53|  Sur-evalue|
|          Garches|   Maison|  150.0|     6266.67|       6402.06|    -2.11| Prix marche|
|  Rueil-malmaison|   Maison| 

In [ ]:
cols = ['ville','type_bien','type_transaction',
        F.round('surface',1).alias('surface'),
        F.round('prix',0).alias('prix'),
        F.round('prix_m2',2).alias('prix_m2_reel'),
        'prix_m2_predit','ecart_pct','score_marche']

print('Top 15 meilleures affaires')
df_scored.filter(F.col('score_marche') == 'Sous-evalue') \
         .orderBy(F.asc('ecart_pct')).select(*cols).show(15, truncate=30)

print('\nTop 15 biens sur-evalues')
df_scored.filter(F.col('score_marche') == 'Sur-evalue') \
         .orderBy(F.desc('ecart_pct')).select(*cols).show(15, truncate=30)

=== Top 15 meilleures affaires ===
+-------------+-----------+----------------+-------+--------+------------+--------------+---------+------------+
|        ville|  type_bien|type_transaction|surface|    prix|prix_m2_reel|prix_m2_predit|ecart_pct|score_marche|
+-------------+-----------+----------------+-------+--------+------------+--------------+---------+------------+
|      Tanques|     Maison|           vente|   87.0| 44000.0|      505.75|       2489.89|   -79.69| Sous-evalue|
|     Boulogne|    inconnu|           vente|   14.0| 10900.0|      778.57|       2803.84|   -72.23| Sous-evalue|
|      Carmaux|     Maison|           vente|  110.0|100000.0|      909.09|       2961.69|   -69.31| Sous-evalue|
|Saint-étienne|Appartement|           vente|   71.0| 69000.0|      971.83|       2918.05|    -66.7| Sous-evalue|
|        Saint|     Maison|           vente|  112.0|120000.0|     1071.43|       2947.91|   -63.65| Sous-evalue|
|        Saint|Appartement|           vente|  118.0|126900.0|

In [ ]:
print('Scoring par ville (min 5 annonces)')
df_scored.groupBy('ville') \
    .agg(
        F.count('*').alias('nb_biens'),
        F.round(F.mean('prix_m2'), 0).alias('prix_m2_moyen'),
        F.round(F.sum(F.when(F.col('score_marche')=='Sous-evalue',1).otherwise(0)) /
                F.count('*') * 100, 1).alias('pct_sous_evalues'),
        F.round(F.sum(F.when(F.col('score_marche')=='Sur-evalue',1).otherwise(0)) /
                F.count('*') * 100, 1).alias('pct_sur_evalues')
    ) \
    .filter(F.col('nb_biens') >= 5) \
    .orderBy(F.desc('pct_sous_evalues')) \
    .show(20, truncate=False)

=== Scoring par ville (min 5 annonces) ===
+-------------------+--------+-------------+----------------+---------------+
|ville              |nb_biens|prix_m2_moyen|pct_sous_evalues|pct_sur_evalues|
+-------------------+--------+-------------+----------------+---------------+
|Tourcoing          |5       |1999.0       |100.0           |0.0            |
|Boulogne           |11      |2052.0       |81.8            |9.1            |
|Créteil            |6       |3730.0       |66.7            |0.0            |
|Antony             |5       |4879.0       |60.0            |0.0            |
|Châtenay-malabry   |5       |5506.0       |60.0            |0.0            |
|Rosny-sous-bois    |7       |3471.0       |57.1            |0.0            |
|Bondy              |6       |3544.0       |50.0            |0.0            |
|Drancy             |7       |3526.0       |42.9            |0.0            |
|Champigny-sur-marne|12      |4165.0       |41.7            |8.3            |
|Saint              |

## 14. Export

In [ ]:
def export_csv(df_spark, filename):
    path = os.path.join(DATA_DIR, filename)
    if os.path.exists(path):
        shutil.rmtree(path) if os.path.isdir(path) else os.remove(path)
    df_spark.toPandas().to_csv(path, index=False, encoding='utf-8')
    print(f'{filename} : {df_spark.count()} lignes exportees')

export_csv(df, 'immobilier_clean.csv')

export_csv(
    df_scored.select(
        'ville','code_postal','departement',
        'type_bien','type_transaction','source',
        F.round('surface',1).alias('surface'),
        F.round('prix',0).alias('prix'),
        'nb_pieces','nb_chambres',
        F.round('prix_m2',2).alias('prix_m2'),
        'prix_m2_predit','ecart_pct','score_marche'
    ),
    'immobilier_scored.csv'
)

immobilier_clean.csv : 821 lignes exportees
immobilier_scored.csv : 821 lignes exportees


## 15. Arrêt Spark

In [ ]:
spark.stop()
print('Spark termine.')

Spark termine.
Fichiers : immobilier_clean.csv | immobilier_scored.csv
